# 캐치에서 바와 가까운 견봉의 위치

허브는 기존 수동점을 사용합니다. 아래 판독 셀은 원본만 보여줍니다. 견봉이 보이는지 먼저 선택한 뒤 클릭하세요. 판단하기 어려우면 판독 불가 버튼을 누르세요. 기존 c04 추정점은 별도로 보존됩니다.

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from IPython.display import display
from wlcv.config import ROOT, VIDEOS
from wlcv.annotation.pose_points import pose_point_editor
from wlcv.annotation.storage import load_events
from wlcv.evaluation.catch_offset import ANNOTATIONS, READS, FACING, shoulder_read, catch_comparison, plot_catches

meta = pd.read_csv(ROOT / "data/metadata/videos.csv").set_index("video_id")
for video in FACING:
    frame = load_events(ANNOTATIONS, video, pass_number=1)["catch"]
    side = meta.at[video, "near_side"]
    saved = shoulder_read(video, frame, side)
    print(video, frame, saved.get("status", "미판독"))
    if not saved or video != "c04_sn_f_01":
        display(pose_point_editor(VIDEOS / f"{video}.mp4", frame, meta.at[video, "person_id"], side,
                                  split="validation", annotations=READS, joints=("어깨",), trunk_review=True))

### 저장한 뒤 비교

40번 실험의 전체 프레임 원판 선택표를 사용합니다. 앞쪽·위쪽이 양수이며 단위는 원본 px입니다. 직접점만 주 평가에 사용합니다. 추정점은 사례로만 표시합니다.

In [ ]:
selected_files = sorted((ROOT / "outputs/phase_speed").glob("*/selected.csv"), key=lambda p: p.stat().st_mtime)
if not selected_files:
    raise FileNotFoundError("먼저 40번 노트북을 실행하세요.")
plates = pd.read_csv(selected_files[-1])
result = catch_comparison(plates)
folder = ROOT / "outputs/catch_offset" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
folder.mkdir(parents=True)
result.to_csv(folder / "offsets.csv", index=False)
display(result[["video_id","frame_index","shoulder_status","pose_status","manual_forward","model_forward","manual_up","model_up"]].round(2))
fig = plot_catches(result)
fig.savefig(folder / "catch.png", dpi=140)